In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import warnings

import arviz_plots as azp
import arviz_stats as azs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from compressor_fouling_modeling.utility import (
    build_bayesian_model,
    build_mixture_baseline,
    calculate_empirical_sigma_stats,
    evaluate_mixture_model,
    fit_bayesian_model,
    get_mixture_residuals,
    plot_loo_calibration_curves,
    plot_mixture_residuals,
    prepare_bayesian_model_args,
    prepare_hierarchical_noise_args,
)

warnings.filterwarnings("ignore")

%config InlineBackend.figure_format = 'retina'  # high resolution figures
azp.style.use("arviz-darkgrid")  # type: ignore

In [ ]:
RANDOM_SEED = 14

In [ ]:
# Define project root relative to notebook location
PROJECT_ROOT = Path().resolve().parents[0]  # goes up one level from /notebooks/
DATA_DIR = PROJECT_ROOT / "data"

X_baseline = pd.read_csv(DATA_DIR / "processed" / "X_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")
y_baseline = pd.read_csv(DATA_DIR / "processed" / "y_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")

In [ ]:
setpoint_unique, setpoint_index, map_sp_to_idx = prepare_hierarchical_noise_args(
    X_baseline
)
empirical_stats, mean_std, range_to_mean_ratio, min_std, max_std = calculate_empirical_sigma_stats(
    X_baseline, y_baseline, setpoint_unique.tolist()
)

print(
    "Prepare the dataset for regressing outlet-pressure residuals against their"
    " respective setpoints."
)
data_residual = prepare_bayesian_model_args(
    X_baseline, y_baseline, shuffle_baseline=False, residual_target=True
)
lr = LinearRegression()
lr.fit(data_residual.X_scaled, data_residual.y_scaled)
y_pred = lr.predict(data_residual.X_scaled)
r2 = r2_score(data_residual.y_scaled, y_pred)
print(
    f"\nRoughly {r2:.3f} of variance can be explained by features altogether after"
    " removing outlet_pressure_sp."
)
plt.hist(data_residual.y_scaled, density=True, bins=25)
plt.title("target (outlet_pressure - outlet_pressure_sp) distribution")
plt.show()

In [ ]:
noise_kwargs = {
    "sigma_mu_mu": mean_std / data_residual.y_std,
    "sigma_mu_sd": np.log(1 + 2.5 * range_to_mean_ratio),  # ~30% variation around
    # mean according to np.exp(sigma_mu_sd)
}
coef_kwargs = {"mu": 0.0, "sd": 0.5}

m_residual = build_bayesian_model(
    X=data_residual.X_scaled,
    y=data_residual.y_scaled,
    intercept_sd=0.5,  # fairly strong shrinkag
    coef_kwargs=coef_kwargs,
    noise_kwargs=noise_kwargs,
    random_seed=RANDOM_SEED
)
idata_residual = fit_bayesian_model(m_residual, random_seed=RANDOM_SEED)

In [ ]:
pc = azp.plot_energy(idata_residual, kind="ecdf")

In [ ]:
pc = azp.combine_plots(
    idata_residual,
    plots=[
        (azp.plot_ppc_dist, {"num_samples": 1000}),
        (azp.plot_ppc_dist, {"num_samples": 1000, "kind": "ecdf"}),
        (azp.plot_loo_pit, {"envelope_prob": 0.95}),
    ],
    group="posterior_predictive",
)
azs.loo(idata_residual)

In [ ]:
# Mixture Model
kwargs = {
    "mu_mu": data_residual.y_scaled.mean(),
    "mu_sigma": 1.0 * data_residual.y_scaled.std(),
    "sigma_sigma": 0.3,
}
# Build and fit
model_mixture = build_mixture_baseline(
    y_deviation=data_residual.y_scaled.to_numpy(),
    kwargs=kwargs,
    n_components=2,
    random_seed=RANDOM_SEED,
)
# When building mixture model (each component has a label), transform=pm.distributions.transforms.Ordered() has been used
# to implicitly sorts the raw draws in the forward pass, mapping them to the ordered space and thus prevent label switching.
# However, PyMC tries to initialize the chain by sampling random jitters.
# If those random initial values happen to be out of order (e.g., component 1 is greater than component 2),
# the log-probability evaluates to -∞, throwing this error.
# To resolve this, one need to provide explicitly sorted initial values to the sampler.
initial_points = {"mu": np.array([-1.0, 1.0])}
idata_mixture = fit_bayesian_model(model_mixture, initvals=initial_points, random_seed=RANDOM_SEED)

In [ ]:
# verify label switching was preveneted
pc = azp.plot_trace(idata_mixture, var_names=["mu"])

In [ ]:
pc = azp.plot_ppc_dist(
    idata_mixture,
    group="prior_predictive",
    kind="ecdf",
    visuals={"predictive_dist": {"color": "C1"}, "observed_dist": {"color": "C3"}},
    num_samples=1000,
)
pc = azp.plot_energy(idata_mixture, kind="ecdf")
pc = azp.plot_ppc_dist(idata_mixture, num_samples=1000)
pc = azp.plot_ppc_dist(idata_mixture, kind="ecdf", num_samples=1000)

In [ ]:
# The sigma overlap is a mild uncertainty about the relative spread of the two
# components, not a sign of misfit or identifiability failure
pc = azp.plot_trace_dist(
    idata_mixture, var_names=["mu", "sigma"], compact=True, combined=False
)

In [ ]:
# pc = azp.plot_loo_pit(idata_mixture, envelope_prob=0.95)
mixture_model_loo = azs.loo(idata_mixture)
print(mixture_model_loo)

In [ ]:
fig_calibration, stats_calibration = plot_loo_calibration_curves(idata_mixture, RANDOM_SEED)

In [ ]:
# two well-separated, stable components with no label switching (no mean overlap)
# and a sensible correlation structure (the data has a fixed overall mean, so if
# one component's mean drifts up, the other tends to follow to compensate.).
pc = azp.plot_pair(
    idata_mixture,
    var_names=["mu"],
    visuals={"divergence": True},
    marginal=True,
    marginal_kind="kde",
)

In [ ]:
fig = evaluate_mixture_model(
    idata_mixture,
    data_residual.y_scaled.to_numpy(),
    y_grid=np.linspace(
        data_residual.y_scaled.min(), data_residual.y_scaled.max(), 1000
    ),
)

In [ ]:
# Get residuals from your fitted mixture model
mixture_model_results = get_mixture_residuals(
    idata_mixture, data_residual.y_scaled.to_numpy()
)
fig_mixture_residuals = plot_mixture_residuals(mixture_model_results)

In [ ]:
# Your standardization parameters
y_mean = data_residual.y_mean  # e.g., 5.018
y_std = data_residual.y_std    # e.g., 2.874

# Convert mu to original scale (deviation from setpoint)
mu_original = [
    -0.666 * y_std + y_mean,  # Low mode
    1.038 * y_std + y_mean,   # High mode
]

print(f"Low Mode:  P ≈ SP + {mu_original[0]:.1f} units")
print(f"High Mode: P ≈ SP + {mu_original[1]:.1f} units")

## System Behavior Analysis

| Metric | Low Mode (60.8% of time) | High Mode (39.2% of time) |
|:-------|:-------------------------:|:-------------------------:|
| **Outlet Pressure** | ~3.1 units **ABOVE** setpoint | ~8.0 units **ABOVE** setpoint |
| **Spread (σ)** | 0.615 standardized<br>≈ 1.8 units original | 0.395 standardized<br>≈ 1.1 units original |
| **Control Characteristic** | Tighter control in this mode | Even tighter control (lower σ) |

### Mode Separation
- **Gap between modes:** 1.698 standardized ≈ 4.9 units original
- **Interpretation:** Very distinct separation between operational modes

### Key Insights
- The system operates in two distinct pressure regimes
- Both modes maintain pressure above setpoint, with High Mode showing ~2.6× higher deviation
- Control tightness (variability) is actually better in High Mode despite higher pressure offset
- The 4.9-unit gap between modes confirms clear bimodal behavior